In [ ]:
## login wandb
import wandb
wandb.login()
## set up project name
import os
os.environ["WANDB_PROJECT"] = "chess-llm" 
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
import unsloth
import vllm
import torch
import trl

print(vllm.__version__)
print(unsloth.__version__)
print(torch.__version__)
print(trl.__version__)

## Model

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 64 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Norrawee/Qwen3-4B-Thinking-2507-exp06", 
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    load_in_fp8 = False, # for new gpus
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

## Data

In [ ]:
from datasets import load_dataset  
  
dataset = load_dataset("Norrawee/grpo-exp07")
df = dataset["train"].to_pandas()
df = df[:100]

In [ ]:
from jinja2 import Template
  
with open("chess_template.jinja") as f:  
    template = Template(f.read())  
  
def format_prompt(row):
    prompt = template.render(
        side_to_move=row["side_to_move"],
        legal_moves_uci=" ".join(row["legal_moves_uci_list"]),
        FEN=row["FEN"],
    )    
    return [  
        {"role": "user", "content": prompt},  
    ]

In [ ]:
## preprocess
df["prompt"] = df.apply(format_prompt, axis=1)

In [ ]:
from datasets import Dataset    
  
# Create HF datasets  
ds = {  
    "train": Dataset.from_pandas(df),  
    "test": Dataset.from_pandas(df[:5]),  
}  

In [ ]:
text = tokenizer.apply_chat_template(
    ds["train"][0]["prompt"], 
    tokenize=False, 
    add_generation_prompt=True,
)
print(len(tokenizer(text)["input_ids"]))
print(text)

## Reward

In [ ]:
import re
UCI_PATTERN = re.compile(r"<uci_move>(.*?)</uci_move>")  
def extract_uci(text):  
    match = UCI_PATTERN.search(text)  
    return match.group(1).strip() if match else None 

In [ ]:
import chess  
import chess.engine  
# ---------------- CONFIG ----------------  
ENGINE_PATH = "stockfish"   # change if needed  
ENGINE_LIMIT = chess.engine.Limit(time=1.0, depth=16) 

In [ ]:
def evaluate(fen_board, uci_move, verbose=False):  
    try:  
        engine = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)  
        board = chess.Board(fen_board)  
  
        player = board.turn  # side making the move  
  
        def score_for_player(pov_score, player):  
            s = pov_score.pov(player)  
            if s.is_mate():  
                mate = s.mate()  
                return 10000 - abs(mate) if mate > 0 else -10000 + abs(mate)  
            return s.score()  
  
        # BEFORE  
        info_before = engine.analyse(board, ENGINE_LIMIT)  
        eval_before = score_for_player(info_before["score"], player)  
  
        # MOVE  
        move = chess.Move.from_uci(uci_move)  
        board.push(move)  
  
        # AFTER  
        info_after = engine.analyse(board, ENGINE_LIMIT)  
        eval_after = score_for_player(info_after["score"], player)  
  
        engine.quit()  
  
        delta = eval_after - eval_before  
  
        reward = delta / 100.0  
        reward = max(-5.0, min(5.0, reward))  # ✅ clip here  
  
        if verbose:  
            print("Before:", eval_before, "After:", eval_after, "Delta:", delta, "Reward:", reward)  
  
        return reward  
  
    except Exception as e:  
        print("Error:", e)  
        return -5.0  

In [ ]:
## Test
fen_board = "k7/r7/8/R7/K7/8/8/8 w - - 0 1" 
uci_move = "a5a6"
x = evaluate(fen_board, uci_move, True)
print(f"White blunders: {x}")

fen_board = "k7/r7/8/R7/K7/8/8/8 b - - 0 1" 
uci_move = "a7a6"  
x = evaluate(fen_board, uci_move, True)
print(f"Black blunders: {x}")

fen_board = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"  
uci_move = "e2e4"  
x = evaluate(fen_board, uci_move, True)  
print(f"Normal move: {x}")  

In [ ]:
## Tag reward
def tag_reward(completions, **kwargs):  
    scores = []
    for completion in completions:
        text = completion[0]["content"]
        ## check uci_move
        uci_move = extract_uci(text)
        if uci_move is not None: 
            scores.append(0.1)
        else: 
            scores.append(0)
    return scores

In [ ]:
## Legal move reward
def legal_reward(completions, legal_moves_uci_list, **kwargs):  
    scores = []
    for completion, legal_moves in zip(completions, legal_moves_uci_list):
        text = completion[0]["content"]
        ## check uci_move
        uci_move = extract_uci(text)
        if uci_move in legal_moves: 
            scores.append(0.1)
        else: 
            scores.append(0)
    return scores

In [ ]:
from concurrent.futures import ProcessPoolExecutor  
  
def move_reward(completions, legal_moves_uci_list, FEN, max_workers=4, **kwargs):  
    tasks = []  
  
    for completion, legal_moves, fen_board in zip(completions, legal_moves_uci_list, FEN):  
        text = completion[0]["content"]  
        uci_move = extract_uci(text)  
  
        if uci_move in legal_moves:  
            tasks.append((fen_board, uci_move))  
        else:  
            tasks.append(None)  
  
    scores = [None] * len(tasks)  
  
    with ProcessPoolExecutor(max_workers=max_workers) as executor:  
        futures = {}  
  
        for i, task in enumerate(tasks):  
            if task is None:  
                scores[i] = -1  
            else:  
                futures[executor.submit(evaluate, task[0], task[1])] = i  
  
        for future in futures:  
            scores[futures[future]] = future.result()  
  
    return scores  

In [ ]:
# Defense   +0.2
# Escape	+0.2
# Fork      +0.5
# Capture   +0.1 - 0.8
# Check	    +0.5
# Mate	    +1.0

PIECE_CAPTURE_BONUS = {  
    chess.PAWN: 0.10,  
    chess.KNIGHT: 0.25,  
    chess.BISHOP: 0.25,  
    chess.ROOK: 0.40,  
    chess.QUEEN: 0.80,  
}  
    
## Extra move reward  
def extra_move_reward(completions, legal_moves_uci_list, FEN, **kwargs):  
    scores = []  
  
    for completion, legal_moves, fen_board in zip(completions, legal_moves_uci_list, FEN):  
        text = completion[0]["content"]  
  
        ## extract uci_move  
        uci_move = extract_uci(text)  
  
        if not uci_move or uci_move not in legal_moves:  
            scores.append(0.0)  
            continue  
  
        # ---------- board states  
        board_before = chess.Board(fen_board)  
        move = chess.Move.from_uci(uci_move)  
  
        board_after = board_before.copy()  
        board_after.push(move)  
  
        bonus = 0.0  
        color = board_before.turn  
  
        # ==================================================  
        # Capture reward (scaled)  
        # ==================================================  
        if board_before.is_capture(move):  
            captured_piece = board_before.piece_at(move.to_square)  
  
            # en passant  
            if captured_piece is None and board_before.is_en_passant(move):  
                captured_piece = chess.Piece(chess.PAWN, not color)  
  
            if captured_piece:  
                bonus += PIECE_CAPTURE_BONUS.get(captured_piece.piece_type, 0.0)  
  
        # ==================================================  
        # Check / Mate  
        # ==================================================  
        if board_after.is_checkmate():  
            scores.append(1.0)  
            continue  
  
        if board_after.is_check():  
            bonus += 0.5
  
        # ==================================================  
        # Escape from attack (moved piece becomes safe)  
        # ==================================================  
        from_sq = move.from_square  
        to_sq = move.to_square  
  
        piece_before = board_before.piece_at(from_sq)  
        piece_after = board_after.piece_at(to_sq)  
  
        if piece_before and piece_after:  
            was_attacked = board_before.is_attacked_by(not color, from_sq)  
            is_safe_now = not board_after.is_attacked_by(not color, to_sq)  
  
            if was_attacked and is_safe_now:  
                bonus += 0.2 
  
        # ==================================================  
        # Fork detection (attacks ≥2 enemy pieces)  
        # ==================================================  
        attacker = piece_after  
        if attacker:  
            attacked = 0  
            for sq in board_after.attacks(to_sq):  
                p = board_after.piece_at(sq)  
                if p and p.color != attacker.color and p.piece_type >= chess.KNIGHT:  
                    attacked += 1  
  
            if attacked >= 2:  
                bonus += 0.5  
  
        # ==================================================  
        # Defense improvement  
        # ==================================================  
        attacked_before = {  
            sq for sq, p in board_before.piece_map().items()  
            if p.color == color and board_before.is_attacked_by(not color, sq)  
        }  
  
        defended = False  
        for sq in attacked_before:  
            if board_after.piece_at(sq) and not board_after.is_attacked_by(not color, sq):  
                defended = True  
                break  
  
        if defended:  
            bonus += 0.2  
  
        # ==================================================  
        # Clip & append  
        # ==================================================  
        bonus = max(0.0, min(5.0, bonus))  
        scores.append(bonus)  
  
    return scores  

In [ ]:
reward_funcs = [tag_reward, legal_reward, move_reward, extra_move_reward]

## GRPO

In [ ]:
max_prompt_length = 400
max_completion_length = max_seq_length - max_prompt_length

# from vllm import SamplingParams
# vllm_sampling_params = SamplingParams(
#     min_p = 0.1,
#     top_k = -1,
#     seed = 3407,
#     stop = [tokenizer.eos_token],
#     include_stop_str_in_output = True,
# )

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    ## algorithm
    importance_sampling_level = "sequence",
    loss_type = "dr_grpo",
    
    ## others
    # vllm_sampling_params = vllm_sampling_params,
    # max_grad_norm = 0.1,
    temperature = 0.9,
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    report_to = "wandb", # Can use Weights & Biases
    
    ## optimizer
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    # beta = 0.00,
    epsilon = 3e-4,
    epsilon_high = 4e-4,
    
    ## training params
    num_generations=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    max_steps=40,
    fp16=False,
    bf16=True,
    
    # logging
    # eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    # eval_steps=5,
    save_total_limit=1,
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs=reward_funcs,
    args = training_args,
    train_dataset = ds["train"],
    # eval_dataset = ds["test"],
)
trainer.train()

## Test

In [ ]:
i = 30
text = tokenizer.apply_chat_template(
    ds["train"][i]["prompt"][:1],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

eos_token = "<|im_end|>"
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0.6,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
    eos_token_id=tokenizer.convert_tokens_to_ids(eos_token),
)

In [ ]:
# Visualize
import chess  
from IPython.display import display, SVG  

fen_board = ds["train"][i]["FEN"]
old_move = ds["train"][i]["target_move"]
new_move = "d1d2"

print(old_move)
evaluate(fen_board, old_move, True)
print(new_move)
evaluate(fen_board, new_move, True)

board = chess.Board(fen_board)
display(SVG(board._repr_svg_()))  

## Save

In [ ]:
# model.push_to_hub_merged(
#     "Norrawee/Qwen3-4B-Thinking-2507-exp07", 
#     tokenizer,
#     save_method = "merged_16bit", 
# )